# OpenNMT-py (Transformer nhỏ, huấn luyện từ đầu) trên dữ liệu song ngữ từng dòng

GPU T4, Internet bật (cài OpenNMT-py vào venv riêng). Không chứa test, không gắn cuộc thi.

In [ ]:
# ===== Cấu hình =====
CAU_HINH = {'huong': ['zh-vi', 'vi-zh'], 'bien_the': ['train'], 'so_epoch': 20, 'seed': 42, 'che_do': 'dich_cum'}

In [ ]:
# Ghi mã vào working
from pathlib import Path
Path('kaggle_onmt.py').write_text('# -*- coding: utf-8 -*-\n"""OpenNMT-py (Transformer nho, HUAN LUYEN TU DAU) tren du lieu song ngu tung dong cua minh. Chay trong notebook Kaggle THUONG (khong gan cuoc thi):\nGPU T4 + Internet BAT (de cai OpenNMT-py 3.5.1 vao venv rieng, torch 2.2.2). Khong doc test; chi doc Dataset ft-song-ngu (pool + mau eval fold 0).\n\nCau hinh mo hinh theo Quang (config/opennmt_A.yaml): Transformer d_model 256, 4 lop, 8 dau, ff 1024, dropout 0,1, batch 4096 token x tich luy 2, Noam lr 2,0,\nbeta2 0,998, label smoothing 0,1, 10 epoch, beam 5, seed 42. Khac: tokenizer SentencePiece unigram 16k moi ngon ngu (Quang chua ghi), khong replace_unk.\nMoi (huong, bien_the) la mot mo hinh: huong zh-vi (1688) va vi-zh (Tiki); bien_the "train" (toan bo train fold 1-4) hoac "khop4m_s42" (mau natural 4M token cua LLM).\nDau ra: onmt/<huong>_<bien_the>/sinh.jsonl ({id, nhom, hyp} dung thu tu eval), onmt_report.json.\n"""\nimport json\nimport math\nimport os\nimport subprocess\nimport sys\nimport time\nfrom collections import Counter\nfrom pathlib import Path\n\nVAO = Path("/kaggle/input")\nRA = Path("/kaggle/working")\nVENV = Path("/tmp/onmt_env")\nPY = str(VENV / "bin" / "python")\n\n\ndef sh(cmd, check=True, tail=2500):\n    print(">>", cmd[:300], flush=True)\n    r = subprocess.run(["bash", "-c", "set -o pipefail; " + cmd], text=True, capture_output=True)\n    print((r.stdout or "")[-tail:], (r.stderr or "")[-tail:], flush=True)\n    if check and r.returncode:\n        raise RuntimeError(f"lenh_loi:{r.returncode}:{cmd[:120]}")\n    return r\n\n\ndef cai():\n    # Kaggle thuong: Ubuntu 24.04, Python 3.13 (venv hong, torch 2.2.2 khong co wheel) -> dung uv de lay Python 3.11\n    sh("pip install -q uv")\n    sh(f"uv venv --python 3.11 {VENV}")\n    sh(f"uv pip install --python {PY} -q \'torch==2.2.2\' \'numpy<2\' \'OpenNMT-py==3.5.1\' sentencepiece")\n    sh(f"{PY} -c \\"import torch, onmt; print(\'torch\', torch.__version__, \'cuda\', torch.cuda.is_available(), \'onmt\', onmt.__version__)\\"")\n\n\ndef tim(ten):\n    ds = [p for p in VAO.rglob(ten) if p.is_file()]\n    if not ds:\n        raise RuntimeError(f"khong_thay:{ten}")\n    return ds[0]\n\n\ndef doc(ten, toi_da=None):\n    L = tim(ten).read_text(encoding="utf-8").split("\\n")\n    if L and L[-1] == "":\n        L = L[:-1]\n    return L[:toi_da] if toi_da else L\n\n\ndef spm_train(files, prefix, vocab=16000):\n    mo = f"{prefix}.model"\n    if Path(mo).exists():\n        return mo\n    sh(f"{PY} -c \\"import sentencepiece as s; s.SentencePieceTrainer.train(input=\'{\',\'.join(files)}\', model_prefix=\'{prefix}\', vocab_size={vocab}, "\n       f"model_type=\'unigram\', character_coverage=0.9995, input_sentence_size=1000000, shuffle_input_sentence=True, unk_id=0, bos_id=-1, eos_id=-1, pad_id=-1, hard_vocab_limit=False)\\"")\n    return mo\n\n\ndef spm_ma(mo, dong, ra):\n    """Ma hoa danh sach dong -> tep cac manh cach nhau bang dau cach; tra danh sach do dai manh."""\n    code = ("import sys, sentencepiece as s, json; sp = s.SentencePieceProcessor(model_file=sys.argv[1]); "\n            "L = json.load(open(sys.argv[2], encoding=\'utf-8\')); "\n            "open(sys.argv[3], \'w\', encoding=\'utf-8\').write(\'\\\\n\'.join(\' \'.join(sp.encode(x, out_type=str)) for x in L) + \'\\\\n\')")\n    tmp = Path(ra).with_suffix(".json")\n    tmp.write_text(json.dumps(dong, ensure_ascii=False), encoding="utf-8")\n    sh(f"{PY} -c \\"{code}\\" {mo} {tmp} {ra}")\n    return [len(x.split()) for x in Path(ra).read_text(encoding="utf-8").split("\\n")[:len(dong)]]\n\n\ndef spm_giai(mo, tep_vao, tep_ra):\n    code = ("import sys, sentencepiece as s; sp = s.SentencePieceProcessor(model_file=sys.argv[1]); "\n            "L = open(sys.argv[2], encoding=\'utf-8\').read().split(\'\\\\n\'); L = L[:-1] if L and L[-1] == \'\' else L; "\n            "open(sys.argv[3], \'w\', encoding=\'utf-8\').write(\'\\\\n\'.join(sp.decode(x.split()) for x in L) + \'\\\\n\')")\n    sh(f"{PY} -c \\"{code}\\" {mo} {tep_vao} {tep_ra}")\n\n\ndef vocab_tu(tep_sp, tep_vocab):\n    c = Counter()\n    for ln in Path(tep_sp).read_text(encoding="utf-8").split("\\n"):\n        c.update(ln.split())\n    Path(tep_vocab).write_text("\\n".join(f"{t}\\t{n}" for t, n in c.most_common()) + "\\n", encoding="utf-8")\n\n\ndef chay_mot(hg, bien_the, cfg, spm, W):\n    src_l, tgt_l = hg.split("-")                      # zh-vi: nguon zh, dich vi\n    t0 = time.time()\n    k = cfg.get("khoi")\n    tr_s, tr_t = doc(f"{hg}.{bien_the}.src", k), doc(f"{hg}.{bien_the}.tgt", k)\n    dv_s, dv_t = doc(f"{hg}.dev.src"), doc(f"{hg}.dev.tgt")\n    ev_s, ev_ids = doc(f"{hg}.eval.src", k), doc(f"{hg}.eval.id", k)\n    ev_nh = doc(f"{hg}.eval.nhom", k)\n    d = W / f"{hg}_{bien_the}"\n    d.mkdir(parents=True, exist_ok=True)\n    ls_tr = spm_ma(spm[src_l], tr_s, d / "train.src")\n    lt_tr = spm_ma(spm[tgt_l], tr_t, d / "train.tgt")\n    spm_ma(spm[src_l], dv_s, d / "dev.src")\n    spm_ma(spm[tgt_l], dv_t, d / "dev.tgt")\n    spm_ma(spm[src_l], ev_s, d / "eval.src")\n    vocab_tu(d / "train.src", d / "vocab.src")\n    vocab_tu(d / "train.tgt", d / "vocab.tgt")\n    # bo cap qua dai (OpenNMT cat theo src_seq_length/tgt_seq_length); dem so dong bi cat de ghi bao cao\n    n_tok = sum(max(a, b) + 2 for a, b in zip(ls_tr, lt_tr))\n    # hieu chinh theo khoi 40.000 dong [DO]: 1 buoc = ~232 cau (batch 4096 token tinh ca padding) = ~3,3k don vi n_tok => buoc/epoch thuc ~ n_tok / 3,3k, gap ~2,2 lan cong thuc ly thuyet\n    steps_epoch = max(1, math.ceil(n_tok / (4096 * 2) * 1.12 * 2.2))\n    # so_epoch la TRAN; dung som theo valid (ppl, kien nhan 5 epoch); luu moi epoch, trung binh 5 checkpoint cuoi\n    steps = int(cfg["so_epoch"]) * steps_epoch if not k else int(cfg.get("khoi_buoc", 200))\n    warm = min(2000, max(300, 2 * steps_epoch)) if not k else max(10, steps // 10)\n    # Noam: lr_dinh = learning_rate * d_model^-0.5 * warmup^-0.5. Giu dinh lr co dinh (2e-3, gan 2,8e-3 cua Quang) bat ke warmup ngan hay dai\n    lr_dinh = float(cfg.get("lr_dinh", 2e-3))\n    lr_noam = lr_dinh * math.sqrt(256 * warm)\n    # ti le title theo token SentencePiece dich (de doi chieu voi 10,3-10,9% cua cac LLM)\n    try:\n        ids_tr = doc(f"{hg}.{bien_the}.id", k)\n        tt = sum(x for x, i in zip(lt_tr, ids_tr) if "_title_" in i)\n        ti_le_title_sp = round(tt / max(1, sum(lt_tr)), 4)\n    except Exception:\n        ti_le_title_sp = None\n    yaml = f"""save_data: {d}/run\nsrc_vocab: {d}/vocab.src\ntgt_vocab: {d}/vocab.tgt\noverwrite: true\ndata:\n  corpus_1:\n    path_src: {d}/train.src\n    path_tgt: {d}/train.tgt\n  valid:\n    path_src: {d}/dev.src\n    path_tgt: {d}/dev.tgt\nsrc_seq_length: 256\ntgt_seq_length: 256\nsave_model: {d}/model\nsave_checkpoint_steps: {150 if k else max(1, min(steps, steps_epoch))}\nkeep_checkpoint: 5\ntrain_steps: {steps}\nvalid_steps: {150 if k else max(1, min(steps, steps_epoch))}\nearly_stopping: {0 if k else 4}\nearly_stopping_criteria: ppl\nwarmup_steps: {warm}\nreport_every: {50 if k else 200}\nworld_size: 1\ngpu_ranks: [0]\nbatch_type: tokens\nbatch_size: 4096\nvalid_batch_size: 2048\naccum_count: [2]\naccum_steps: [0]\noptim: adam\nlearning_rate: {lr_noam:.4f}\ndecay_method: noam\nadam_beta2: 0.998\nlabel_smoothing: 0.1\nparam_init: 0.0\nparam_init_glorot: true\nencoder_type: transformer\ndecoder_type: transformer\nposition_encoding: true\nenc_layers: 4\ndec_layers: 4\nheads: 8\nhidden_size: 256\nword_vec_size: 256\ntransformer_ff: 1024\ndropout: [0.1]\nattention_dropout: [0.1]\nseed: {int(cfg[\'seed\'])}\n"""\n    (d / "train.yaml").write_text(yaml, encoding="utf-8")\n    t1 = time.time()\n    sh(f"{VENV}/bin/onmt_train -config {d}/train.yaml > {d}/train.log 2>&1", check=False, tail=100)\n    r = sh(f"grep -E \'Step [0-9]+/|Validation|Early|Stop\' {d}/train.log | grep -v Weighted | awk \'NR%1==0\' | tail -n 40", check=False, tail=6000)\n    ck = sorted(d.glob("model_step_*.pt"), key=lambda p: int(p.stem.split("_")[-1]))\n    if not ck:\n        raise RuntimeError(f"khong_co_checkpoint:{hg}:{bien_the}")\n    giay_train = time.time() - t1\n    mo_hinh_dich, trung_binh = ck[-1], False\n    if len(ck) >= 2:\n        rr = sh(f"{VENV}/bin/onmt_average_models -models {\' \'.join(str(p) for p in ck)} -output {d}/avg.pt", check=False)\n        if (d / "avg.pt").exists():\n            mo_hinh_dich, trung_binh = d / "avg.pt", True\n    sh(f"{VENV}/bin/onmt_translate -model {mo_hinh_dich} -src {d}/eval.src -output {d}/eval.hyp.sp -beam_size 5 -batch_size 64 -batch_type sents -gpu 0 -max_length 256 -seed 42", check=False)\n    spm_giai(spm[tgt_l], d / "eval.hyp.sp", d / "eval.hyp")\n    hyp = (d / "eval.hyp").read_text(encoding="utf-8").split("\\n")[:len(ev_s)]\n    (d / "sinh.jsonl").write_text("\\n".join(json.dumps({"id": i, "nhom": g, "hyp": h}, ensure_ascii=False) for i, g, h in zip(ev_ids, ev_nh, hyp)), encoding="utf-8")\n    sh(f"cp {d}/train.log {RA}/onmt/{hg}_{bien_the}_train.log", check=False, tail=100)\n    for tep in ("train.src", "train.tgt", "dev.src", "dev.tgt", "eval.src", "eval.hyp.sp", "run*", "model_step_*.pt"):\n        for p in d.glob(tep):\n            p.unlink()\n    return {"huong": hg, "bien_the": bien_the, "dong_train": len(tr_s), "token_sp_train": n_tok, "steps": steps, "steps_epoch": steps_epoch, "warmup": warm,\n            "lr_dinh": lr_dinh, "learning_rate_noam": round(lr_noam, 4), "ti_le_title_token_sp_dich": ti_le_title_sp,\n            "so_checkpoint": len(ck), "checkpoint_cuoi": ck[-1].name, "trung_binh_checkpoint": trung_binh, "buoc_da_chay": int(ck[-1].stem.split("_")[-1]),\n            "giay_train": round(giay_train, 1), "giay_tong": round(time.time() - t0, 1), "n_eval": len(ev_s), "checkpoint": mo_hinh_dich.name,\n            "log_cuoi": (r[1] if isinstance(r, tuple) else (r.stdout or ""))[-2500:]}\n\n\ndef dich_cum(cfg):\n    """Dich eval ban cum (tung manh) bang mo hinh OpenNMT da huan luyen (avg.pt + spm trong dau ra notebook nlp-onmt, gan qua kernel_sources); khong huan luyen lai."""\n    t0 = time.time()\n    cai()\n    W = RA / "onmt_cum"\n    W.mkdir(parents=True, exist_ok=True)\n    spm = {l: str(tim(f"spm_{l}.model")) for l in ("zh", "vi")}\n    bc = {"cau_hinh": cfg, "viec": []}\n    for hg in cfg["huong"]:\n        sl, tl = hg.split("-")\n        ck = [p for p in VAO.rglob("avg.pt") if p.parent.name == f"{hg}_train"]\n        if not ck:\n            raise RuntimeError(f"khong_thay_avg.pt:{hg}")\n        d = W / hg\n        d.mkdir(exist_ok=True)\n        src, ids = doc(f"{hg}.evalcum.src"), doc(f"{hg}.evalcum.id")\n        spm_ma(spm[sl], src, d / "cum.src")\n        sh(f"{VENV}/bin/onmt_translate -model {ck[0]} -src {d}/cum.src -output {d}/cum.hyp.sp -beam_size 5 -batch_size 64 -batch_type sents -gpu 0 -max_length 256 -seed 42", check=False)\n        spm_giai(spm[tl], d / "cum.hyp.sp", d / "cum.hyp")\n        hyp = (d / "cum.hyp").read_text(encoding="utf-8").split("\\n")[:len(src)]\n        gom = {}\n        for i, h in zip(ids, hyp):\n            gom.setdefault(i.rsplit("#", 1)[0], []).append(h)\n        nhom = {json.loads(l)["id"]: json.loads(l)["nhom"] for l in tim(f"{hg}.evalcum_items.jsonl").read_text(encoding="utf-8").split("\\n") if l.strip()}\n        (d / "sinh_cum.jsonl").write_text("\\n".join(json.dumps({"id": i, "nhom": nhom.get(i, ""), "hyp": "; ".join(v)}, ensure_ascii=False) for i, v in gom.items()), encoding="utf-8")\n        bc["viec"].append({"huong": hg, "n_manh": len(src), "n_mau": len(gom), "checkpoint": str(ck[0])})\n        for p in (d / "cum.src", d / "cum.hyp.sp"):\n            p.unlink()\n    bc["giay_tong"] = round(time.time() - t0, 1)\n    (RA / "onmt_cum_report.json").write_text(json.dumps(bc, ensure_ascii=False, indent=1), encoding="utf-8")\n    print("XONG dich cum", len(bc["viec"]), flush=True)\n\n\ndef main(cfg):\n    if cfg.get("che_do") == "dich_cum":\n        return dich_cum(cfg)\n    t0 = time.time()\n    cai()\n    W = RA / "onmt"\n    W.mkdir(parents=True, exist_ok=True)\n    k = cfg.get("khoi")\n    # SentencePiece: mot mo hinh cho moi ngon ngu, hoc tu CA HAI huong cua train day du (ben zh: nguon zh-vi + dich vi-zh; ben vi: dich zh-vi + nguon vi-zh)\n    tam = W / "spm_data"\n    tam.mkdir(exist_ok=True)\n    (tam / "zh.txt").write_text("\\n".join(doc("zh-vi.train.src", k) + doc("vi-zh.train.tgt", k)) + "\\n", encoding="utf-8")\n    (tam / "vi.txt").write_text("\\n".join(doc("zh-vi.train.tgt", k) + doc("vi-zh.train.src", k)) + "\\n", encoding="utf-8")\n    spm = {"zh": spm_train([str(tam / "zh.txt")], str(W / "spm_zh")), "vi": spm_train([str(tam / "vi.txt")], str(W / "spm_vi"))}\n    bc = {"cau_hinh": cfg, "viec": []}\n    for hg in cfg["huong"]:\n        for bt in cfg["bien_the"]:\n            try:\n                bc["viec"].append(chay_mot(hg, bt, cfg, spm, W))\n            except Exception as e:\n                import traceback\n                bc["viec"].append({"huong": hg, "bien_the": bt, "loi": f"{type(e).__name__}: {e}", "traceback": traceback.format_exc()[-2500:]})\n                print("LOI", hg, bt, e, flush=True)\n            (RA / "onmt_report.json").write_text(json.dumps(bc, ensure_ascii=False, indent=1), encoding="utf-8")\n    bc["giay_tong"] = round(time.time() - t0, 1)\n    (RA / "onmt_report.json").write_text(json.dumps(bc, ensure_ascii=False, indent=1), encoding="utf-8")\n    print("XONG", len(bc["viec"]), flush=True)\n\n\nif __name__ == "__main__":\n    raise SystemExit("Nhung vao notebook, goi main(CAU_HINH)")\n', encoding='utf-8')


In [ ]:
import sys
sys.path.insert(0, '.')
import kaggle_onmt
kaggle_onmt.main(CAU_HINH)
